# Advanced Problems: Named Tuples for Returning Multiple Values

This notebook contains advanced practice problems with complete solutions.

Topic focus:

- Returning multiple values from functions
- Replacing plain tuples with `namedtuple` results
- Improving readability with named fields
- Preserving tuple unpacking behavior
- Designing stable function return types
- Validating returned data
- Using `_asdict()`, `_replace()`, and dynamic field access


In [1]:
from collections import namedtuple
from random import randint, random, seed
from statistics import mean
from math import sqrt

## Problem 1: Replace a Plain Tuple Return with a Named Tuple

A function currently returns an RGBA color as a plain tuple:

```python
return red, green, blue, alpha
```

Rewrite the function so that it returns a named tuple called `Color`.

Requirements:

1. The named tuple should have fields `red`, `green`, `blue`, and `alpha`.
2. The returned object should still support tuple unpacking.
3. The returned object should allow field access using dot notation.
4. `red`, `green`, and `blue` should be integers between `0` and `255`.
5. `alpha` should be a float between `0` and `1`, rounded to two decimal places.


### Solution 1

In [2]:
Color = namedtuple('Color', 'red green blue alpha')


def random_color():
    """
    Return a random RGBA color as a named tuple.
    """
    red = randint(0, 255)
    green = randint(0, 255)
    blue = randint(0, 255)
    alpha = round(random(), 2)

    return Color(red, green, blue, alpha)

In [3]:
seed(10)

color = random_color()
color

Color(red=16, green=219, blue=247, alpha=0.58)

In [4]:
color.red, color.green, color.blue, color.alpha

(16, 219, 247, 0.58)

In [5]:
red, green, blue, alpha = color

print(f'red={red}, green={green}, blue={blue}, alpha={alpha}')

red=16, green=219, blue=247, alpha=0.58


### Key lesson

A named tuple behaves like a regular tuple, so unpacking still works.

The advantage is that fields can also be accessed by meaningful names, such as `color.red` instead of `color[0]`.

## Problem 2: Prevent Misuse of Positional Indexes

A plain tuple return value can make code difficult to understand:

```python
result = analyze_numbers([10, 20, 30])
result[0]
result[1]
result[2]
```

Write a function called `analyze_numbers` that returns a named tuple called `NumberStats` with these fields:

- `count`
- `minimum`
- `maximum`
- `average`

Requirements:

1. Raise a `ValueError` if the input sequence is empty.
2. Return a `NumberStats` instance.
3. Demonstrate both index-based access and named-field access.
4. Explain why named-field access is better for readability.


### Solution 2

In [6]:
NumberStats = namedtuple('NumberStats', 'count minimum maximum average')


def analyze_numbers(numbers):
    """
    Return basic statistics for a non-empty sequence of numbers.
    """
    if not numbers:
        raise ValueError('numbers must not be empty')

    return NumberStats(
        count=len(numbers),
        minimum=min(numbers),
        maximum=max(numbers),
        average=mean(numbers)
    )

In [7]:
stats = analyze_numbers([10, 20, 30, 40, 50])
stats

NumberStats(count=5, minimum=10, maximum=50, average=30)

In [8]:
# Positional access works, but it is not very descriptive.
stats[0], stats[1], stats[2], stats[3]

(5, 10, 50, 30)

In [9]:
# Named access is much clearer.
stats.count, stats.minimum, stats.maximum, stats.average

(5, 10, 50, 30)

### Key lesson

`stats.average` communicates intent immediately.

`stats[3]` forces the reader to remember the meaning of position `3`.

## Problem 3: Return Validation Results from a Function

Write a function called `validate_color` that accepts four values: `red`, `green`, `blue`, and `alpha`.

The function should return a named tuple called `ValidationResult` with these fields:

- `is_valid`
- `errors`
- `color`

Requirements:

1. `red`, `green`, and `blue` must be integers from `0` to `255`.
2. `alpha` must be an integer or float between `0` and `1`.
3. If the input is valid, `color` should contain a `Color` instance.
4. If the input is invalid, `color` should be `None`.
5. `errors` should be a tuple of error messages.


### Solution 3

In [10]:
ValidationResult = namedtuple('ValidationResult', 'is_valid errors color')


def validate_color(red, green, blue, alpha):
    """
    Validate RGBA color components and return a structured result.
    """
    errors = []

    components = {
        'red': red,
        'green': green,
        'blue': blue
    }

    for name, value in components.items():
        if not isinstance(value, int):
            errors.append(f'{name} must be an integer')
        elif not 0 <= value <= 255:
            errors.append(f'{name} must be between 0 and 255')

    if not isinstance(alpha, (int, float)):
        errors.append('alpha must be a number')
    elif not 0 <= alpha <= 1:
        errors.append('alpha must be between 0 and 1')

    if errors:
        return ValidationResult(False, tuple(errors), None)

    color = Color(red, green, blue, round(alpha, 2))
    return ValidationResult(True, tuple(), color)

In [11]:
valid_result = validate_color(120, 80, 200, 0.75)
valid_result

ValidationResult(is_valid=True, errors=(), color=Color(red=120, green=80, blue=200, alpha=0.75))

In [12]:
invalid_result = validate_color(300, 'green', 50, 1.5)
invalid_result

ValidationResult(is_valid=False, errors=('red must be between 0 and 255', 'green must be an integer', 'alpha must be between 0 and 1'), color=None)

### Key lesson

Returning a named tuple makes the function's result self-documenting.

Instead of guessing what `result[0]`, `result[1]`, and `result[2]` mean, we can write:

```python
result.is_valid
result.errors
result.color
```

## Problem 4: Return Multiple Related Results from a Geometry Function

Write a function called `distance_report` that accepts two points:

```python
p1 = (x1, y1)
p2 = (x2, y2)
```

The function should return a named tuple called `DistanceReport` with these fields:

- `dx`
- `dy`
- `distance`
- `midpoint`

Requirements:

1. `dx` should be `x2 - x1`.
2. `dy` should be `y2 - y1`.
3. `distance` should be the Euclidean distance.
4. `midpoint` should be another named tuple called `Point`.
5. The return value should support unpacking and named-field access.


### Solution 4

In [13]:
Point = namedtuple('Point', 'x y')
DistanceReport = namedtuple('DistanceReport', 'dx dy distance midpoint')


def distance_report(p1, p2):
    """
    Return distance-related information between two 2D points.
    """
    p1 = Point(*p1)
    p2 = Point(*p2)

    dx = p2.x - p1.x
    dy = p2.y - p1.y
    distance = sqrt(dx ** 2 + dy ** 2)
    midpoint = Point(
        x=(p1.x + p2.x) / 2,
        y=(p1.y + p2.y) / 2
    )

    return DistanceReport(dx, dy, distance, midpoint)

In [14]:
report = distance_report((0, 0), (3, 4))
report

DistanceReport(dx=3, dy=4, distance=5.0, midpoint=Point(x=1.5, y=2.0))

In [15]:
report.distance, report.midpoint.x, report.midpoint.y

(5.0, 1.5, 2.0)

In [16]:
dx, dy, distance, midpoint = report

dx, dy, distance, midpoint

(3, 4, 5.0, Point(x=1.5, y=2.0))

### Key lesson

Named tuples can be nested.

Here, `DistanceReport` contains a `Point` instance as its `midpoint` field.

## Problem 5: Design a Stable Return Type for a Function

Suppose a function originally returns this plain tuple:

```python
(success, value, error)
```

This is fragile because callers must remember what each position means.

Create a named tuple called `OperationResult` and write a function called `safe_divide`.

Requirements:

1. If division succeeds, return `OperationResult(True, value, None)`.
2. If division fails because of division by zero, return `OperationResult(False, None, error_message)`.
3. Demonstrate how named fields make the return value easier to use.
4. Convert the result to a dictionary using `_asdict()`.


### Solution 5

In [17]:
OperationResult = namedtuple('OperationResult', 'success value error')


def safe_divide(a, b):
    """
    Safely divide two numbers and return a structured operation result.
    """
    try:
        value = a / b
    except ZeroDivisionError:
        return OperationResult(False, None, 'Cannot divide by zero')

    return OperationResult(True, value, None)

In [18]:
success_result = safe_divide(10, 2)
failure_result = safe_divide(10, 0)

success_result, failure_result

(OperationResult(success=True, value=5.0, error=None),
 OperationResult(success=False, value=None, error='Cannot divide by zero'))

In [19]:
if success_result.success:
    print(f'Result: {success_result.value}')
else:
    print(f'Error: {success_result.error}')

Result: 5.0


In [20]:
failure_result._asdict()

{'success': False, 'value': None, 'error': 'Cannot divide by zero'}

### Key lesson

Named tuples are useful for result objects because they communicate intent without requiring a custom class.

## Problem 6: Update a Returned Named Tuple Without Mutation

Named tuples are immutable.

Use `_replace()` to create a modified copy of a returned color.

Write a function called `with_full_opacity` that accepts a `Color` and returns a new `Color` with `alpha=1.0`.

Requirements:

1. Do not mutate the original color.
2. Use `_replace()`.
3. Show that the original and updated colors are different objects.


### Solution 6

In [21]:
def with_full_opacity(color):
    """
    Return a copy of color with alpha set to 1.0.
    """
    return color._replace(alpha=1.0)

In [22]:
original = Color(10, 20, 30, 0.25)
updated = with_full_opacity(original)

original, updated

(Color(red=10, green=20, blue=30, alpha=0.25),
 Color(red=10, green=20, blue=30, alpha=1.0))

In [23]:
original is updated

False

### Key lesson

`_replace()` creates a new named tuple instance.

The original instance remains unchanged.

## Problem 7: Dynamic Access to Returned Fields

Sometimes the field you want to access is stored in a variable.

For example:

```python
field = 'green'
```

Write a helper function called `get_field` that:

1. Accepts a named tuple instance.
2. Accepts a field name.
3. Accepts an optional default value.
4. Returns the field value if the field exists.
5. Returns the default if the field does not exist.


### Solution 7

In [24]:
def get_field(record, field_name, default=None):
    """
    Safely retrieve a field from a named tuple using a dynamic field name.
    """
    return getattr(record, field_name, default)

In [25]:
color = Color(100, 150, 200, 0.8)

field = 'green'
get_field(color, field)

150

In [26]:
get_field(color, 'brightness', default='field does not exist')

'field does not exist'

### Key lesson

Use `getattr(record, field_name, default)` when the field name is stored in a variable.

`record.field_name` looks for a literal field named `field_name`, which is usually not what you want.

## Problem 8: Advanced Challenge — Return a Rich Color Analysis Object

Write a function called `analyze_color` that accepts a `Color` and returns a named tuple called `ColorAnalysis`.

`ColorAnalysis` should contain:

- `color`
- `rgb_tuple`
- `hex_value`
- `is_transparent`
- `brightness`

Requirements:

1. `rgb_tuple` should contain only `(red, green, blue)`.
2. `hex_value` should be a string like `'#0A141E'`.
3. `is_transparent` should be `True` when `alpha < 1`.
4. `brightness` should be the average of red, green, and blue.
5. Return a named tuple, not a dictionary.


### Solution 8

In [27]:
ColorAnalysis = namedtuple(
    'ColorAnalysis',
    'color rgb_tuple hex_value is_transparent brightness'
)


def analyze_color(color):
    """
    Return a rich analysis object for a Color named tuple.
    """
    rgb_tuple = (color.red, color.green, color.blue)
    hex_value = f'#{color.red:02X}{color.green:02X}{color.blue:02X}'
    is_transparent = color.alpha < 1
    brightness = mean(rgb_tuple)

    return ColorAnalysis(
        color=color,
        rgb_tuple=rgb_tuple,
        hex_value=hex_value,
        is_transparent=is_transparent,
        brightness=brightness
    )

In [28]:
analysis = analyze_color(Color(10, 20, 30, 0.5))
analysis

ColorAnalysis(color=Color(red=10, green=20, blue=30, alpha=0.5), rgb_tuple=(10, 20, 30), hex_value='#0A141E', is_transparent=True, brightness=20)

In [29]:
analysis.hex_value, analysis.is_transparent, analysis.brightness

('#0A141E', True, 20)

## Problem 9: Compare Plain Tuple Returns and Named Tuple Returns

Write two functions:

1. `plain_color()` returns a plain tuple.
2. `named_color()` returns a `Color` named tuple.

Then compare:

- Representation
- Readability
- Unpacking
- Attribute access


### Solution 9

In [30]:
def plain_color():
    return 12, 34, 56, 0.7


def named_color():
    return Color(12, 34, 56, 0.7)

In [31]:
plain = plain_color()
named = named_color()

plain, named

((12, 34, 56, 0.7), Color(red=12, green=34, blue=56, alpha=0.7))

In [32]:
# Both support unpacking.
r1, g1, b1, a1 = plain
r2, g2, b2, a2 = named

(r1, g1, b1, a1), (r2, g2, b2, a2)

((12, 34, 56, 0.7), (12, 34, 56, 0.7))

In [33]:
# Only the named tuple supports meaningful attribute access.
named.red, named.green, named.blue, named.alpha

(12, 34, 56, 0.7)

### Key lesson

A named tuple keeps the lightweight behavior of a tuple while making the return value much easier to understand.

## Summary of Best Practices

1. Use plain tuples for very small, obvious, local return values.
2. Use named tuples when returned values have clear meanings.
3. Define the named tuple type once, not every time the function is called.
4. Use keyword arguments when constructing named tuples for clarity.
5. Preserve unpacking compatibility when replacing plain tuples.
6. Use dot notation for readable client code.
7. Use `_asdict()` when a dictionary representation is needed.
8. Use `_replace()` to create modified copies.
9. Use `getattr()` for dynamic field access.
10. Prefer structured return types for validation, parsing, analysis, and operation results.
